# Evaluate a Local LLM with Ollama

This notebook turns one local-model run into a small, reproducible product evaluation. You will distinguish cold and warm behavior, use Ollama's own usage fields, and compare model settings without treating fluent text as proof of quality.

## Learning Goals

- verify the local runtime and exact model tag;
- interpret total, load, prompt-evaluation, and generation durations;
- calculate generation throughput from output tokens and generation time;
- compare one factor at a time; and
- preserve evidence for a local, cloud, or hybrid recommendation.

Run the notebook from top to bottom. Ollama must be running and `qwen3.5:2b` must be installed for live generation cells. If the service is unavailable, the notebook will explain what was skipped instead of failing.

## 1. Import the Tools

The standard library handles timing, summaries, and statistics. `requests` communicates with Ollama's local HTTP API. Imports remain explicit so the environment requirements are visible.

In [ ]:
import json
import statistics
import time
from typing import Any

import requests

## 2. Configure and Check the Runtime

Use an explicit model tag so another participant can reproduce the same setup. The health check asks Ollama for its installed model tags and does not generate text.

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"
GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"
MODEL_NAME = "qwen3.5:2b"
REQUEST_TIMEOUT_SECONDS = 120
MAX_OUTPUT_TOKENS = 128

session = requests.Session()


def get_installed_models() -> tuple[bool, list[str]]:
    """Return the Ollama status and installed model names."""
    try:
        response = session.get(TAGS_URL, timeout=3)
        response.raise_for_status()
    except requests.RequestException:
        return False, []

    models = [item.get("name", "") for item in response.json().get("models", [])]
    return True, [name for name in models if name]


ollama_available, installed_models = get_installed_models()
print(f"Ollama available: {ollama_available}")
print(f"Installed models: {installed_models or 'none detected'}")

model_available = ollama_available and MODEL_NAME in installed_models
if ollama_available and not model_available:
    print(f"Install the course model with: ollama pull {MODEL_NAME}")
if not ollama_available:
    print("Start the Ollama application or local service before live evaluation.")

## 3. Normalize Ollama Metrics

Ollama reports durations in nanoseconds. Generation throughput uses `eval_count / eval_duration`, not the complete wall-clock duration. The parser can be tested with a fixed response even when no local model is running.

![Ollama API usage fields](assets/ollama-api-metrics.png)

*The official API documents the token counts and duration fields returned by Ollama. Source: [Ollama API usage documentation](https://docs.ollama.com/api/usage).*

In [ ]:
def seconds(nanoseconds: float | None) -> float:
    """Convert an Ollama duration from nanoseconds to seconds."""
    return float(nanoseconds or 0) / 1_000_000_000


def normalize_metrics(data: dict[str, Any], wall_seconds: float) -> dict[str, Any]:
    """Select and calculate the metrics used in the evaluation."""
    generation_seconds = seconds(data.get("eval_duration"))
    output_tokens = int(data.get("eval_count", 0))

    return {
        "model": data.get("model", "unknown"),
        "text": data.get("response", ""),
        "wall_seconds": wall_seconds,
        "total_seconds": seconds(data.get("total_duration")),
        "load_seconds": seconds(data.get("load_duration")),
        "prompt_seconds": seconds(data.get("prompt_eval_duration")),
        "generation_seconds": generation_seconds,
        "prompt_tokens": int(data.get("prompt_eval_count", 0)),
        "output_tokens": output_tokens,
        "output_tokens_per_second": (
            output_tokens / generation_seconds if generation_seconds > 0 else 0.0
        ),
    }


sample_response = {
    "model": MODEL_NAME,
    "response": "Example response",
    "total_duration": 2_000_000_000,
    "load_duration": 500_000_000,
    "prompt_eval_count": 20,
    "prompt_eval_duration": 200_000_000,
    "eval_count": 30,
    "eval_duration": 1_000_000_000,
}
sample_metrics = normalize_metrics(sample_response, wall_seconds=2.1)
assert sample_metrics["output_tokens_per_second"] == 30.0
sample_metrics

## 4. Send a Non-Streaming Request

This helper sends one complete request with a timeout and records both client wall time and Ollama's internal metrics. Because `stream` is false, it does **not** measure time to first token. The exercise disables Qwen3.5's optional reasoning mode and limits output length so short comparisons complete predictably.

In [ ]:
def query_model(
    prompt: str,
    *,
    temperature: float = 0.2,
    model: str = MODEL_NAME,
) -> dict[str, Any]:
    """Generate one response and return normalized text and metrics."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "think": False,
        "options": {
            "temperature": temperature,
            "num_predict": MAX_OUTPUT_TOKENS,
        },
    }

    started = time.perf_counter()
    response = session.post(
        GENERATE_URL,
        json=payload,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    wall_seconds = time.perf_counter() - started
    response.raise_for_status()
    return normalize_metrics(response.json(), wall_seconds)


def metric_summary(result: dict[str, Any]) -> dict[str, Any]:
    """Return a compact view without reproducing the complete model output."""
    keys = (
        "model",
        "wall_seconds",
        "total_seconds",
        "load_seconds",
        "prompt_tokens",
        "output_tokens",
        "output_tokens_per_second",
    )
    return {key: result[key] for key in keys}

## 5. Establish Cold and Warm Behavior

The first generation may load the model into memory. Later generations can reuse the loaded model. Keep both observations because either can affect the user experience.

Replace the example with a synthetic input representative of your chosen product task.

In [ ]:
baseline_prompt = (
    "Summarize this product feedback in one sentence: "
    "The export is useful, but it takes too long and sometimes loses selected filters."
)

baseline_runs: list[dict[str, Any]] = []
if model_available:
    for run_number in range(3):
        result = query_model(baseline_prompt, temperature=0.2)
        result["run_number"] = run_number + 1
        baseline_runs.append(result)
        print(json.dumps(metric_summary(result), indent=2))
else:
    print("Live baseline skipped because the configured model is unavailable.")

### Interpret the Baseline

Treat the first run as a possible cold request and the later runs as warm requests. Verify the `load_seconds` values instead of assuming. Median warm performance is more informative than selecting the fastest run.

In [ ]:
if len(baseline_runs) >= 3:
    warm_total_seconds = [run["total_seconds"] for run in baseline_runs[1:]]
    warm_throughput = [run["output_tokens_per_second"] for run in baseline_runs[1:]]
    baseline_summary = {
        "first_run_total_seconds": baseline_runs[0]["total_seconds"],
        "first_run_load_seconds": baseline_runs[0]["load_seconds"],
        "median_warm_total_seconds": statistics.median(warm_total_seconds),
        "median_warm_output_tokens_per_second": statistics.median(warm_throughput),
    }
    print(json.dumps(baseline_summary, indent=2))
else:
    print("Run three live requests to calculate the cold/warm summary.")

## 6. Compare One Factor at a Time

Define quality criteria before running the experiment. The examples below compare temperature while keeping the model and prompt fixed. Repeat each condition because one response cannot establish reliable behavior.

Adapt the prompt to your product task and preserve only non-sensitive evidence.

In [ ]:
evaluation_prompt = (
    "Classify this synthetic support request as billing, technical, or account. "
    "Return only the category. Request: I cannot reset my password."
)
temperatures = (0.0, 0.8)
repetitions = 2

experiment_runs: list[dict[str, Any]] = []
if model_available:
    for temperature in temperatures:
        for repetition in range(1, repetitions + 1):
            result = query_model(evaluation_prompt, temperature=temperature)
            result.update(
                {
                    "temperature": temperature,
                    "repetition": repetition,
                }
            )
            experiment_runs.append(result)
            print(
                f"temperature={temperature}, repetition={repetition}, "
                f"output={result['text']!r}"
            )
else:
    print("Live comparison skipped because the configured model is unavailable.")

## 7. Score Quality with Human-Owned Criteria

For the classification example, useful criteria could be:

| Criterion | Score rule |
|---|---|
| Correct category | `1` only if the expected category is returned |
| Exact format | `1` only if no explanation or extra text is included |
| Stability | Compare whether repeated runs return the same valid result |

Create criteria appropriate to your task. Record the scores beside the observed outputs; do not replace judgment with a vague impression of fluency.

In [ ]:
def exact_category_score(text: str, expected: str = "account") -> int:
    """Score the synthetic classification response using an exact rule."""
    return int(text.strip().lower() == expected)


if experiment_runs:
    for run in experiment_runs:
        run["exact_category_score"] = exact_category_score(run["text"])

    scored_results = [
        {
            "temperature": run["temperature"],
            "repetition": run["repetition"],
            "text": run["text"].strip(),
            "exact_category_score": run["exact_category_score"],
            "total_seconds": round(run["total_seconds"], 3),
        }
        for run in experiment_runs
    ]
    print(json.dumps(scored_results, indent=2, ensure_ascii=True))
else:
    print("Run the live experiment before scoring model output.")

## 8. Turn Measurements into a Product Decision

Review the collected evidence, then interpret it cautiously:

- Results describe this model, device, configuration, and test set.
- A high tokens-per-second value does not prove output quality.
- A correct example does not establish reliability across the use case.
- A `localhost` endpoint does not prove the complete application is private.
- Non-streaming total duration is not time to first token.

A technical conclusion should connect representative quality, cold and warm performance, the complete data flow, hardware constraints, and failure impact.

## Next: Streaming and Structured Output

The next notebook implements streaming time to first token and schema-validated JSON output. These techniques make local-model behavior easier to integrate into responsive, testable software.

## References

- [Ollama API: Generate a Response](https://docs.ollama.com/api/generate)
- [Ollama API: Streaming](https://docs.ollama.com/api/streaming)
- [Ollama API: Usage Metrics](https://docs.ollama.com/api/usage)
- [Ollama Model Library: Qwen3.5](https://ollama.com/library/qwen3.5/tags)